# シミュレーション計算実行用ノートブック

## 1. 今回の実験の説明

In [1]:
DESCRIPTION = '''
mlflowを使ってシミュレーションプログラムの実行とその結果を管理する実験の例題2。SPH6のシミュレーションを行う。
'''
ISSUE_NO = ''
EXEC_NAME = 'SPH6'
RUN_SCRIPT = "calcSPH6"

MLFLOW_EXP_TYPE = "particleSPH_NS_S3"

dump_files = ["dump.SPH6", "dump.SPH6box"]

## 2. シミュレーションパラメータ

In [2]:
import importlib, json
sim = importlib.import_module(f'scripts.{RUN_SCRIPT}')

## units used in this simulation are
## [g][cm][s]
sim_prams = sim.default_prams

## 必要なら適宜修正する
sim_prams["stepmax"] = 0.5


print(json.dumps(sim_prams, indent=4, ensure_ascii=False))

{
    "cell": [
        0.0,
        20.0,
        0.0,
        20.0,
        0.0,
        25.0
    ],
    "lunit": 0.4,
    "sph_h_factor": 0.5938629,
    "rho0_coef": 20.0,
    "mu_fluid": 0.0089,
    "mu_solid": 100000.0,
    "c_fluid": 750.0,
    "c_solid": 2720.0,
    "deltaT": 5e-05,
    "stepmax": 0.5,
    "intaval": 0.005,
    "param_g": 980.0
}


checking GPU devices
GPU status: no error
found 1 GPU
  Device 0: Tesla T4
Tesla T4 (compute capability 7.5)
  number of multiprocessor: 40
  number of cores / MP: 64
  global memory size: 2.5632 [GB]
  max threads per block: 1024
  max block per grid: 2147483647
  shared memory size: 48 [KB]


## 3. mlflow変数

In [3]:
import mlflow
import os
from time import strftime, gmtime

In [ ]:
## 1台構成の時
#MLFLOW_TRACKING_URI = "sqlite:////home/ubuntu/mlruns/mlflow.db"
#MLFLOW_STORAGE = "file:///home/ubuntu/mlstorage"
### S3 bucket を指定する場合
MLFLOW_STORAGE = "s3://my-mlflow-artifact-s3-bucket/mlstorage/"

## mlflow serverのIPを指定
MLFLOW_TRACKING_URI = "http://localhost:5000"


## github, backlogなどでチケット管理をしている場合はそのBASE URLを設定
ISSUE_BASE_URL = 'https://xxxxx/'

### dump fileの圧縮に使うコマンド（pixz があれば推奨）
ARCHIVE_COMMAND = "pixz"

###
### mlflow変数　自動設定
###
MYNAME = os.getenv("USER")
#GIT_INFO = gitutils.get_info()
RUN_NAME = EXEC_NAME + strftime("-%Y-%m-%d-%H-%M-%S", gmtime())

if ISSUE_NO != '':
    ISSUE_NAME = f'\n[{ISSUE_NO}]({ISSUE_BASE_URL}{ISSUE_NO})'
else:
    ISSUE_NAME = ''


## 4. シミュレーション実行

In [5]:
###
### mlflow処理開始
###
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_exp = mlflow.get_experiment_by_name(MLFLOW_EXP_TYPE)
if mlflow_exp is None:
    mlflow_exp_id = mlflow.create_experiment(name=MLFLOW_EXP_TYPE, artifact_location=MLFLOW_STORAGE)
else:
    mlflow_exp_id = mlflow_exp.experiment_id

In [6]:
mlflow_run = mlflow.start_run(
    experiment_id=mlflow_exp_id,
    run_name=RUN_NAME,
    description=f'{DESCRIPTION}{ISSUE_NAME}')

print(f"Run ID: {mlflow_run.info.run_id}")

mlflow.set_tag("mlflow.user", MYNAME)
mlflow.set_tag("simulation", EXEC_NAME)
mlflow.set_tag("run_script", RUN_SCRIPT)
#mlflow.log_params({'git_commit': GIT_INFO['commit'], 'git_branch': GIT_INFO['branch']})
#mlflow.log_artifact('git.diff.txt', artifact_path='git_info')

Run ID: 748dfe8cbf9a4ac3afd24e8f1e176253


In [ ]:
batch_script = f'''#! /bin/sh
#SBATCH -J {RUN_NAME}
#SBATCH -o {RUN_NAME}.out
#SBATCH -e {RUN_NAME}.err
#SBATCH -n 1
#SBATCH -t 00:00:00

export RUN_SCRIPT={RUN_SCRIPT}
export MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}
export RUN_NAME={RUN_NAME}
export ARCHIVE_COMMAND={ARCHIVE_COMMAND}
# JSON 文字列はシェルで空白区切りされないようにクォートする
export SIM_PARAMS='{json.dumps(sim_prams)}'
export DUMP_FILES='{json.dumps(dump_files)}'

python3 Run.py {mlflow_run.info.run_id}
'''

import subprocess

# sbatch コマンドに batch_script を標準入力で渡して実行
proc = subprocess.run(
    ["sbatch"],
    input=batch_script,
    text=True,
    capture_output=True
)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)


Submitted batch job 6

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
                 6      main SPH6-202   ubuntu PD       0:00      1 (None)


0

In [ ]:
os.system(f'squeue -u {MYNAME}')

In [ ]:
## 異常・中断時の mlflow.end_run()
run_info = mlflow.get_run(mlflow_run.info.run_id)
if run_info.info.lifecycle_stage == "active":
    mlflow.end_run(status='KILLED')
